# Data Exploration — Análises Adicionais
Complementa o notebook principal de Data Exploration.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

STOP         = set(stopwords.words('english'))
CLASS_COLORS = {0: '#e74c3c', 1: '#2ecc71', 2: '#3498db'}
CLASS_NAMES  = {0: 'Bearish (0)', 1: 'Bullish (1)', 2: 'Neutral (2)'}
plt.rcParams['figure.dpi'] = 120

# Carregar dados
df = pd.read_csv('train.csv')

# Função de limpeza base (igual ao preprocessing)
def tokenize_clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\$[a-zA-Z]{1,5}', 'ticker', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = text.split()
    return [t for t in tokens if t not in STOP and len(t) > 2]

print(f'Dataset carregado: {df.shape}')
print('✅ Imports OK')

## 1. Análise de Sentimento com Palavras Financeiras

In [ ]:
# ── Palavras financeiras por classe ───────────────────────────────────────────
financial_words = {
    'Bearish': ['bear', 'bearish', 'crash', 'dump', 'fall', 'drop', 'loss',
                'decline', 'miss', 'misses', 'cut', 'lower', 'down', 'sell',
                'short', 'recession', 'fear', 'risk', 'warning'],
    'Bullish': ['bull', 'bullish', 'rally', 'rise', 'gain', 'surge', 'high',
                'beat', 'beats', 'raised', 'higher', 'buy', 'long', 'growth',
                'profit', 'record', 'boom', 'up', 'strong', 'outperform'],
    'Neutral': ['report', 'result', 'earn', 'announce', 'declare', 'say',
                'conference', 'meeting', 'call', 'present', 'quarter']
}

# Contar ocorrências de cada palavra por classe
results = []
for category, words in financial_words.items():
    for word in words:
        for lbl in [0, 1, 2]:
            count = df[df['label'] == lbl]['text'].str.lower().str.contains(
                rf'\b{word}\b', regex=True).sum()
            results.append({'category': category, 'word': word,
                            'class': CLASS_NAMES[lbl], 'count': count})

res_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (category, color) in zip(axes, [('Bearish','#e74c3c'),
                                          ('Bullish','#2ecc71'),
                                          ('Neutral','#3498db')]):
    cat_df = res_df[res_df['category'] == category]
    pivot  = cat_df.pivot(index='word', columns='class', values='count').fillna(0)
    pivot.plot(kind='barh', ax=ax,
               color=[CLASS_COLORS[0], CLASS_COLORS[1], CLASS_COLORS[2]],
               edgecolor='none', alpha=0.85)
    ax.set_title(f'Palavras {category}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Nº de tweets')
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)
    ax.legend(fontsize=8)

plt.suptitle('Distribuição de palavras financeiras por classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_financial_words.png', bbox_inches='tight')
plt.show()

print('CONCLUSÃO: As palavras financeiras aparecem maioritariamente na classe esperada?')
print('→ Se sim, o dataset é coerente e estas palavras serão boas features.')

## 2. Tweets Duplicados

In [ ]:
# ── Duplicados ─────────────────────────────────────────────────────────────────
dup_exatos  = df.duplicated(subset='text').sum()
dup_label   = df.duplicated(subset=['text', 'label']).sum()

print(f'Total de tweets:                        {len(df):,}')
print(f'Duplicados (texto igual):               {dup_exatos}')
print(f'Duplicados (texto + label igual):       {dup_label}')

# Caso mais perigoso: mesmo tweet, labels diferentes
mask      = df.duplicated(subset='text', keep=False)
conflitos = df[mask].groupby('text')['label'].nunique()
conflitos = conflitos[conflitos > 1]
print(f'Tweets com labels contraditórias:       {len(conflitos)}')

if dup_exatos > 0:
    print(f'\nExemplos de tweets duplicados:')
    print(df[mask][['text','label']].sort_values('text').head(8).to_string())
else:
    print('\n✅ Sem duplicados — dataset limpo.')

## 3. Análise de N-gramas (Bigramas e Trigramas)

In [ ]:
# ── N-gramas por classe ────────────────────────────────────────────────────────
from nltk.util import ngrams

def get_ngrams(texts, n):
    all_ngrams = []
    for text in texts:
        tokens = tokenize_clean(text)
        all_ngrams.extend([' '.join(g) for g in ngrams(tokens, n)])
    return Counter(all_ngrams)

for n, label_n in [(2, 'Bigramas'), (3, 'Trigramas')]:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for lbl, ax in zip([0, 1, 2], axes):
        ng = get_ngrams(df[df['label'] == lbl]['text'], n)
        top = ng.most_common(15)
        if top:
            words, counts = zip(*top)
            ax.barh(list(reversed(words)), list(reversed(counts)),
                    color=CLASS_COLORS[lbl], edgecolor='none', alpha=0.85)
        ax.set_title(f'{CLASS_NAMES[lbl]}', fontsize=11,
                     fontweight='bold', color=CLASS_COLORS[lbl])
        ax.set_xlabel('Frequência')
        ax.grid(axis='x', alpha=0.3)
        ax.spines[['top','right']].set_visible(False)
    plt.suptitle(f'Top 15 {label_n} por classe', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'fig_{label_n.lower()}.png', bbox_inches='tight')
    plt.show()

print('CONCLUSÃO: Vês bigramas como "beats earnings" no Bullish e "misses estimates" no Bearish?')
print('→ Se sim, vale a pena usar ngram_range=(1,2) no TF-IDF.')

## 4. Comprimento dos Tweets por Classe (Boxplot)

In [ ]:
# ── Comprimento por classe ─────────────────────────────────────────────────────
df['n_words'] = df['text'].str.split().str.len()
df['n_chars'] = df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for metric, ax, title in [
    ('n_words', axes[0], 'Nº de palavras por tweet'),
    ('n_chars',  axes[1], 'Nº de caracteres por tweet')
]:
    data   = [df[df['label'] == lbl][metric].values for lbl in [0, 1, 2]]
    bp     = ax.boxplot(data, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 2})
    for patch, lbl in zip(bp['boxes'], [0, 1, 2]):
        patch.set_facecolor(CLASS_COLORS[lbl])
        patch.set_alpha(0.7)
    ax.set_xticklabels(['Bearish', 'Bullish', 'Neutral'], fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig_length_boxplot.png', bbox_inches='tight')
plt.show()

# Teste estatístico para ver se a diferença é significativa
from scipy import stats
bear = df[df['label'] == 0]['n_words']
bull = df[df['label'] == 1]['n_words']
neut = df[df['label'] == 2]['n_words']

stat, p = stats.kruskal(bear, bull, neut)
print(f'Kruskal-Wallis test (diferença de comprimento entre classes):')
print(f'  p-value = {p:.4f}')
if p < 0.05:
    print('  → Diferença estatisticamente significativa! O comprimento pode ser uma feature útil.')
else:
    print('  → Sem diferença significativa. O comprimento não discrimina bem as classes.')

print(f'\nMédia de palavras por classe:')
print(df.groupby('label')['n_words'].mean().round(1).rename(index=CLASS_NAMES))

## 5. Tweets sem Conteúdo Útil após Limpeza

In [ ]:
# ── Tweets vazios ou muito curtos após limpeza ────────────────────────────────
df['tokens_clean'] = df['text'].apply(tokenize_clean)
df['n_tokens_clean'] = df['tokens_clean'].str.len()

vazios       = (df['n_tokens_clean'] == 0).sum()
muito_curtos = (df['n_tokens_clean'] < 3).sum()

print(f'Tweets vazios após limpeza (0 tokens):      {vazios}')
print(f'Tweets com menos de 3 tokens:               {muito_curtos}')
print(f'Tweets com conteúdo útil (≥3 tokens):       {(df["n_tokens_clean"] >= 3).sum():,}')

# Distribuição do nº de tokens limpos
fig, ax = plt.subplots(figsize=(10, 4))
for lbl in [0, 1, 2]:
    subset = df[df['label'] == lbl]['n_tokens_clean']
    ax.hist(subset, bins=30, alpha=0.55,
            color=CLASS_COLORS[lbl], label=CLASS_NAMES[lbl], edgecolor='none')
ax.axvline(x=3, color='red', linestyle='--', linewidth=1.5, label='Mínimo (3 tokens)')
ax.set_title('Distribuição de tokens úteis após limpeza', fontsize=12, fontweight='bold')
ax.set_xlabel('Nº de tokens (após limpeza)')
ax.set_ylabel('Frequência')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig_tokens_after_clean.png', bbox_inches='tight')
plt.show()

if muito_curtos > 0:
    print(f'\nExemplos de tweets problemáticos (< 3 tokens após limpeza):')
    mask = df['n_tokens_clean'] < 3
    print(df[mask][['text', 'label', 'n_tokens_clean']].head(10).to_string())

## 6. Análise Temporal (se existir coluna de data)

In [ ]:
# ── Análise temporal ──────────────────────────────────────────────────────────
date_cols = [c for c in df.columns if any(kw in c.lower()
             for kw in ['date', 'time', 'created', 'timestamp'])]

if date_cols:
    date_col = date_cols[0]
    print(f'Coluna de data encontrada: {date_col}')
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df['month']  = df[date_col].dt.to_period('M')

    pivot = df.groupby(['month', 'label']).size().unstack(fill_value=0)
    pivot.columns = [CLASS_NAMES[c] for c in pivot.columns]

    fig, ax = plt.subplots(figsize=(14, 5))
    pivot.plot(kind='bar', ax=ax,
               color=[CLASS_COLORS[0], CLASS_COLORS[1], CLASS_COLORS[2]],
               edgecolor='none', alpha=0.85)
    ax.set_title('Distribuição de tweets por mês e classe', fontsize=13, fontweight='bold')
    ax.set_xlabel('Mês')
    ax.set_ylabel('Nº de tweets')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('fig_temporal.png', bbox_inches='tight')
    plt.show()
else:
    print('Sem coluna de data no dataset — análise temporal não aplicável.')

## 7. Palavras Exclusivas de Cada Classe

In [ ]:
# ── Vocabulário exclusivo por classe ──────────────────────────────────────────
vocab_per_class = {}
freq_per_class  = {}

for lbl in [0, 1, 2]:
    tokens = []
    for text in df[df['label'] == lbl]['text']:
        tokens.extend(tokenize_clean(text))
    vocab_per_class[lbl] = set(tokens)
    freq_per_class[lbl]  = Counter(tokens)

# Palavras que aparecem APENAS numa classe
for lbl in [0, 1, 2]:
    others  = set().union(*[vocab_per_class[l] for l in [0,1,2] if l != lbl])
    exclusive = vocab_per_class[lbl] - others
    # ordenar por frequência
    top_excl = sorted(exclusive, key=lambda w: freq_per_class[lbl][w], reverse=True)[:20]
    print(f'\n{CLASS_NAMES[lbl]} — Top 20 palavras exclusivas ({len(exclusive)} no total):')
    for w in top_excl:
        print(f'  {w:<20} {freq_per_class[lbl][w]:>4}')

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for lbl, ax in zip([0, 1, 2], axes):
    others   = set().union(*[vocab_per_class[l] for l in [0,1,2] if l != lbl])
    exclusive = vocab_per_class[lbl] - others
    top_excl  = sorted(exclusive, key=lambda w: freq_per_class[lbl][w], reverse=True)[:15]
    counts    = [freq_per_class[lbl][w] for w in top_excl]
    ax.barh(list(reversed(top_excl)), list(reversed(counts)),
            color=CLASS_COLORS[lbl], edgecolor='none', alpha=0.85)
    ax.set_title(f'Exclusivas — {CLASS_NAMES[lbl]}',
                 fontsize=11, fontweight='bold', color=CLASS_COLORS[lbl])
    ax.set_xlabel('Frequência')
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Palavras exclusivas de cada classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_exclusive_words.png', bbox_inches='tight')
plt.show()

## 8. Dispersão Lexical

In [ ]:
# ── Dispersão lexical ─────────────────────────────────────────────────────────
# Para cada palavra: em quantos tweets diferentes aparece?
from collections import defaultdict

word_doc_count = defaultdict(int)  # nº de tweets onde a palavra aparece
word_total     = Counter()          # nº total de ocorrências

for text in df['text']:
    tokens = set(tokenize_clean(text))  # set = conta 1 vez por tweet
    for t in tokens:
        word_doc_count[t] += 1

for text in df['text']:
    word_total.update(tokenize_clean(text))

# Top 30 palavras mais frequentes — ver dispersão
top30 = [w for w, _ in word_total.most_common(30)]

disp_df = pd.DataFrame({
    'palavra'     : top30,
    'total_ocorr' : [word_total[w] for w in top30],
    'n_tweets'    : [word_doc_count[w] for w in top30],
})
disp_df['ocorr_por_tweet'] = (disp_df['total_ocorr'] / disp_df['n_tweets']).round(2)
disp_df['pct_tweets']      = (disp_df['n_tweets'] / len(df) * 100).round(1)

print('Top 30 palavras — frequência total vs dispersão em tweets:')
print(f"{'Palavra':<18} {'Total':>8} {'Nº tweets':>10} {'% tweets':>10} {'Ocorr/tweet':>12}")
print('-' * 62)
for _, row in disp_df.iterrows():
    print(f"{row['palavra']:<18} {int(row['total_ocorr']):>8,} "
          f"{int(row['n_tweets']):>10,} {row['pct_tweets']:>9.1f}% "
          f"{row['ocorr_por_tweet']:>12.2f}")

# Scatter: total de ocorrências vs nº de tweets
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(disp_df['n_tweets'], disp_df['total_ocorr'],
           color='#2c3e50', s=80, alpha=0.8, edgecolors='white')
for _, row in disp_df.iterrows():
    ax.annotate(row['palavra'],
                (row['n_tweets'], row['total_ocorr']),
                fontsize=8, ha='left', va='bottom',
                xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('Nº de tweets onde a palavra aparece')
ax.set_ylabel('Total de ocorrências')
ax.set_title('Dispersão Lexical — Top 30 palavras', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig_lexical_dispersion.png', bbox_inches='tight')
plt.show()

print('\nCONCLUSÃO:')
print('Palavras próximas da diagonal (ocorr/tweet ≈ 1) aparecem uma vez por tweet → boas features.')
print('Palavras muito acima da diagonal aparecem várias vezes no mesmo tweet → podem ser ruído.')

## Resumo das Conclusões

In [ ]:
print('=' * 65)
print('RESUMO DAS ANÁLISES ADICIONAIS')
print('=' * 65)
print(f'1. Palavras financeiras  → [COMPLETAR após ver os gráficos]')
print(f'2. Duplicados            → {df.duplicated(subset="text").sum()} encontrados')
print(f'3. N-gramas              → [COMPLETAR — há bigramas distintivos por classe?]')
print(f'4. Comprimento           → [COMPLETAR — p-value do Kruskal-Wallis]')
print(f'5. Tweets vazios         → {(df["n_tokens_clean"] == 0).sum()} vazios, '
      f'{(df["n_tokens_clean"] < 3).sum()} com < 3 tokens')
print(f'6. Temporal              → [N/A se não houver coluna de data]')
print(f'7. Palavras exclusivas   → [COMPLETAR — quantas por classe?]')
print(f'8. Dispersão lexical     → [COMPLETAR — palavras concentradas ou dispersas?]')
print('=' * 65)